# Nottingham Forest full-back and wing-back validation

This notebook validates the side-specific and role-normalised wide-defender templates before they are added to the application. It checks coordinate orientation, sampling uncertainty, within-player stability, sensitivity to zone layout and sensitivity to action selection.

Run `scripts/validate_forest_wide_defenders.py` first. Generated tables remain local because they derive from licensed event data.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'

def load_role(stem):
    return {
        'summary': pd.read_csv(PROCESSED_ROOT / f'forest_{stem}_validation_summary.csv'),
        'bootstrap': pd.read_csv(PROCESSED_ROOT / f'forest_{stem}_bootstrap_summary.csv'),
        'split_half': pd.read_csv(PROCESSED_ROOT / f'forest_{stem}_split_half_summary.csv'),
        'layout': pd.read_csv(PROCESSED_ROOT / f'{stem}_layout_robustness.csv'),
        'families': pd.read_csv(PROCESSED_ROOT / f'forest_{stem}_action_family_sensitivity.csv'),
        'quality': pd.read_csv(PROCESSED_ROOT / f'forest_{stem}_validation_quality_checks.csv'),
    }

full_back = load_role('full_back')
wing_back = load_role('wing_back')

## 1. Quality and orientation checks

Side-specific profiles must remain on their original side. Role-normalised profiles must appear on the left because right-sided coordinates are mirrored onto the left. All failure counts should be zero.

In [ ]:
pd.concat(
    [full_back['quality'].assign(role='Full-back'), wing_back['quality'].assign(role='Wing-back')],
    ignore_index=True,
).pivot(index='check', columns='role', values='value')

## 2. Forest sample eligibility

A player needs five eligible appearances in the selected template group to contribute to that league template. Profiles below five appearances remain visible but provisional.

In [ ]:
samples = pd.concat([full_back['summary'], wing_back['summary']], ignore_index=True)
(samples[[
    'position_group', 'template_group', 'player_name',
    'eligible_appearances', 'template_eligible'
]]
 .drop_duplicates()
 .sort_values(['position_group', 'template_group', 'eligible_appearances'], ascending=[True, True, False]))

## 3. Qualified Forest full-back comparisons

Neco Williams, Ola Aina and Nicolò Savona qualify for the overarching full-back comparison. Williams qualifies as a left full-back; Aina and Savona qualify as right full-backs. Similarity describes spatial conventionality, not performance quality.

In [ ]:
full_back['summary'].loc[
    full_back['summary']['template_eligible'],
    [
        'player_name', 'template_group', 'layout', 'eligible_appearances',
        'similarity_score', 'relative_lateral_shift', 'orientation_alignment'
    ],
].round(2)

## 4. Bootstrap uncertainty and split-half stability

Bootstrap intervals resample complete appearances while holding the league baseline fixed. Split-half estimates require at least ten appearances. Savona qualifies for the template but has only nine appearances, so his wider bootstrap interval should be used instead of a split-half estimate.

In [ ]:
qualified_keys = full_back['summary'].loc[
    full_back['summary']['template_eligible'],
    ['player_id', 'template_group', 'layout'],
].drop_duplicates()
(full_back['bootstrap']
 .merge(qualified_keys, on=['player_id', 'template_group', 'layout'])
 [['player_name', 'template_group', 'layout', 'appearance_count',
   'similarity_score', 'similarity_ci_low', 'similarity_ci_high']]
 .round(2))

In [ ]:
full_back['split_half'].loc[
    full_back['split_half']['split_half_status'].eq('calculated'),
    ['player_name', 'template_group', 'layout', 'appearance_count',
     'median_split_half_similarity', 'split_half_p05', 'split_half_p95'],
].round(2)

## 5. Zone-layout sensitivity

The full-back layouts show broad agreement, but correlations are lower than the centre-back result. The layouts therefore support the same general interpretation but should remain separate analytical views rather than being treated as interchangeable measurements.

In [ ]:
pd.concat(
    [full_back['layout'].assign(position_group='Full-back'),
     wing_back['layout'].assign(position_group='Wing-back')],
    ignore_index=True,
).round(3)

## 6. Action-family sensitivity

Passing and defensive actions produce meaningfully different spatial similarities. This supports retaining separate action-family views when the role is added to the application.

In [ ]:
family_data = full_back['families'].loc[
    full_back['families']['template_group'].eq('Full-back')
    & full_back['families']['eligible_family_appearances'].ge(5)
]
family_data.pivot_table(
    index=['player_name', 'layout'],
    columns='action_family',
    values='similarity_score',
).round(1)

## 7. Wing-back decision

The league wing-back templates and coordinate transformations pass validation, but every Forest wing-back profile has fewer than five eligible appearances. None can support split-half validation. These profiles may be displayed later as explicitly provisional exploratory results, but they should not currently be used for firm tactical conclusions.

## Decision

The full-back pipeline is sufficiently stable to add to the application with side-specific and overarching comparisons. Neco Williams and Ola Aina have the strongest within-player stability; Savona is eligible but should retain an uncertainty cue because he has nine appearances. The wing-back league templates are valid, but Forest comparisons must remain provisional. Both zone layouts and separate action-family views should be preserved.